<a href="https://colab.research.google.com/github/c-capocasali/Search-Engine/blob/main/PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho Final - SME0142 - Álgebra Linear e Aplicações
## Implementação do Algoritmo PageRank para artigos da Wikipédia

**Autores:**
*   Caio Capocasali nºUSP 12541733
*   Otávio Biagioni Melo nºUSP 15482604

**Vídeo Apresentação:** https://www.youtube.com/watch?v=A7-bPxzz4wU

**Sobre o Projeto:**

O projeto está dividido em duas partes principais:
*   Coleta de Dados (Web Scraping)
*   Aplicação do PageRank (Álgebra Linear)

## Parte 1: Coleta de Dados



In [ ]:
# Instalar bibliotecas não-padrões
!pip install bidict tldextract

In [ ]:
# --- IMPORTS E CONFIGURAÇÃO INICIAL ---
import requests
import numpy as np
from scipy.sparse import csr_matrix, lil_matrix
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from bidict import bidict
import tldextract
import json
import sys
import urllib3
import time
import re
from collections import deque

# Desabilita avisos de segurança para conexões HTTPS não verificadas
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

### Funcionamento da classe WebGraphCrawler
A classe funciona a partir de um **URL inicial**. Ela extrai um número (*max_links_per_page*) de links presentes nessa página e os coloca em uma fila *deque* para processamento subsequente. O parâmetro *max_depth* limita a profundidade desse rastreamento.

**Durante a execução:**
* Cada URL passa por *get_clean_url* e é validado por *is_valid_url*.
* O método *get_or_create_id* cria ou recupera um ID inteiro que representa o URL, armazenado no bidicionário *self.urls*.
* As relações entre páginas são registradas em *self.adjacency_list*, onde cada chave é um ID e o valor é o conjunto de IDs apontados por ele.
* O texto da página é processado para construir *self.inverted_index*, que mapeia cada palavra para a lista de IDs das páginas nas quais ela aparece.

In [ ]:
class WebGraphCrawler:
    def __init__(self):
        # Mapeia ID (int) <-> URL (str)
        self.urls = bidict()

        # Mapeia ID (int) -> Set de IDs vizinhos {int, int...}
        self.adjacency_list = {}

        # Mapeia Palavra (str) -> Set de páginas onde ela ocorre {int, int...}
        self.inverted_index = {}

        self.counter = 0
        self.visited = set()

    def get_or_create_id(self, url):
        """Dado URL, retorna o ID se existir ou então cria um novo."""
        # URL já identificada
        if url in self.urls.inv:
            return self.urls.inv[url]
        # Nova URL/ID
        new_id = self.counter
        self.urls[new_id] = url
        self.adjacency_list[new_id] = set()
        self.counter += 1
        return new_id

    def is_valid_url(self, url):
        """Filtra URLs inválidas ou que não sejam http/https."""
        bl = ["Wikip%C3%A9dia:", "Wikipedia:", "Ficheiro:", "Ajuda:",
              "Categoria:", "Portal:", "Especial:", "Talk:",
              "Predefinição:", "Wikimedia_Foundation",
              "Licen%C3%A7a_livre", "Wikip%C3%A9dia",
              "Creative_Commons", "index.php"]
        p = urlparse(url)
        if not (p.scheme in ('http', 'https') and p.netloc):
            return False
        if 'wikipedia.org' not in url:
            return False
        return not any(b in url for b in bl)

    def get_clean_url(self, url):
        """Retorna a URL higienizada."""
        return url.split('#')[0].split('?')[0].rstrip('/')

    def crawl(self, start_url, max_depth=2, max_links_per_page=20):
        """Função principal do Crawler."""
        # Higieniza URL inicial
        start_url = self.get_clean_url(start_url)
        if not self.is_valid_url(start_url):
            print("URL inicial inválida.")
            return

        # Fila para gerenciar URLs: (url_atual, profundidade_atual)
        queue = deque([(start_url, 0)])

        # Marca a URL inicial como visitada
        self.visited.add(start_url)

        # Header do Crawler
        headers = {'User-Agent': 'Mozilla/5.0 (compatible; GraphCrawler/1.0)'}

        while queue:
            url, current_depth = queue.popleft()

            current_id = self.get_or_create_id(url)
            print(f"Depth {current_depth}: Crawling {url} (ID: {current_id})")

            # Evita erro "403 - Too Many Requests"
            time.sleep(0.2)

            try:
                response = requests.get(url, headers=headers, timeout=5, verify=False, stream=True)

                # Verificações antes de baixar o HTML
                if response.status_code != 200:
                    response.close()
                    continue

                content_type = response.headers.get('Content-Type', '').lower()
                if 'text/html' not in content_type:
                    response.close()
                    continue

                # Lê apenas 1MB
                html_content = response.content[:1024 * 1024]
                soup = BeautifulSoup(html_content, 'html.parser')

                # Limpeza do DOM
                for trash in soup.select('header,nav,script,style,aside,iframe,form,footer'):
                    trash.decompose()

                search_area = soup.select_one('main,article,#content,body') or soup

                # -- Indexação de palavras --
                text = search_area.get_text(separator=' ', strip=True).lower()
                words_on_page = set(re.findall(r'\b\w{3,}\b', text))

                for word in words_on_page:
                    if word not in self.inverted_index:
                        self.inverted_index[word] = set()
                    self.inverted_index[word].add(current_id)

                # -- Extração de links apontados --
                if current_depth >= max_depth:
                    continue

                # Pega todos os links únicos da página raw
                unique_raw_links = {a['href'] for a in search_area.find_all('a', href=True)}

                links_found_count = 0

                for raw_href in unique_raw_links:
                    if links_found_count >= max_links_per_page:
                        break

                    # Filtra tags <a> não-HTML
                    if raw_href.startswith(('#', 'javascript:', 'mailto:', 'tel:')):
                        continue

                    full_url = urljoin(url, raw_href)
                    clean_full_url = self.get_clean_url(full_url)

                    # Filtros lógicos
                    if clean_full_url == url: continue
                    if not self.is_valid_url(clean_full_url): continue

                    target_id = self.get_or_create_id(clean_full_url)

                    # Adiciona aresta no grafo
                    if target_id in self.adjacency_list[current_id]:
                        continue

                    self.adjacency_list[current_id].add(target_id)
                    links_found_count += 1

                    # Adiciona a fila
                    if clean_full_url not in self.visited:
                        self.visited.add(clean_full_url)
                        queue.append((clean_full_url, current_depth + 1))

            except Exception as e:
                print(f"Erro ao processar {url}: {str(e)}")

    def save_graph(self, filename='web_graph.json'):
        """Salva as estruturas de dado para um json."""
        # Converte Sets para Listas para serialização JSON
        adj_list_serializable = {
            k: sorted(list(v)) for k, v in self.adjacency_list.items()
        }

        inverted_index_serializable = {
            k: sorted(list(v)) for k, v in self.inverted_index.items()
        }

        data = {
            "id_to_url": dict(self.urls),
            "adjacency_list": adj_list_serializable,
            "inverted_index": inverted_index_serializable
        }

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        print(f"Grafo salvo em {filename}")

    def load_graph(self, filename='web_graph.json'):
        """Carrega de um json para as estruturas de dado."""
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)

                self.urls = bidict({int(k): v for k, v in data["id_to_url"].items()})

                # Carrega listas e converte de volta para sets
                self.adjacency_list = {int(k): set(v) for k, v in data["adjacency_list"].items()}
                self.inverted_index = {k: set(v) for k, v in data.get("inverted_index", {}).items()}

                # Restaura contador e visitados
                self.counter = max(self.urls.keys()) + 1 if self.urls else 0
                self.visited = set(self.urls.values())

            print(f"Grafo carregado de {filename}. Total URLs: {len(self.urls)}")
            return self.urls, self.adjacency_list, self.inverted_index
        except FileNotFoundError:
            print("Arquivo não encontrado.")
            return None, None

In [ ]:
# --- EXECUÇÃO PRINCIPAL DO CRAWLER ---
crawler = WebGraphCrawler()

# Escolha aqui a URL Inicial:
start_url = "https://pt.wikipedia.org/wiki/Brasil"

print("Iniciando crawler...")

# Altere aqui a profundidade de busca e links máximos por página:
crawler.crawl(start_url, max_depth=3, max_links_per_page=15)

crawler.save_graph('grafo.json')

Iniciando crawler...
Depth 0: Crawling https://pt.wikipedia.org/wiki/Brasil (ID: 0)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Copa_do_Mundo_FIFA_de_1994 (ID: 1)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Gasolina (ID: 2)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Ainda_Estou_Aqui_(filme_de_2024) (ID: 3)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Produ%C3%A7%C3%A3o_de_caf%C3%A9_no_Brasil (ID: 4)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Arquitetura_neocl%C3%A1ssica (ID: 5)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Talian (ID: 6)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Baixada_Santista (ID: 7)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Tacac%C3%A1 (ID: 8)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Organiza%C3%A7%C3%A3o_dos_Estados_Americanos (ID: 9)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Pol%C3%ADtica_do_Brasil (ID: 10)
Depth 1: Crawling https://pt.wikipedia.org/wiki/Comando-Geral_de_Tecnologia_Aeroespacial (ID: 11)
Depth 1: Cra

## Parte 2: Aplicação do PageRank

O PageRank é um algoritmo utilizado para prever qual seria o comportamento do usuário enquanto navega pela web. Para resolver esse problema, é necessário modelar uma cadeia de Markov da redes, no qual os vértices são os sites e as arestas os hyperlinks. Cada site inicia com uma probabilidade (estado), que representa a probabilidade do usuário estar presente no site.

O objetivo principal é encontrar o estado estacionário da cadeia, ou seja, um momento $t$ tal que $t -> \infty$ em que não há mudanças significativas nos estados da cadeia.

### Matriz de Probabilidade

O trecho implementa a probabilidade inicial de localização do usuário. Para garantir uma distribuição justa, assumimos que a chance de começar em qualquer site é igual para todos.

Já para a construção do grafo, os pesos das conexões foram definidos com base na quantidade de hyperlinks que cada site possui.

No caso do site não possuir nenhum hyperlink (dead-end), estabelecemos que ele será conectado com todos os outros sites, simulando o comportamento do usuário.

In [ ]:
def prob_matrix(graph):
    P = []
    n_sites = len(graph)

    rows = []
    cols = []
    data = []

    #Percorre o dicionário (lista de adjacência)
    for site, links in graph.items():
        n_links = len(links)
        tmp = np.zeros(n_sites)

        #Se o site não referenciar nenhum outro (dead end), devemos
        #assumir que o usuário pode clicar em qualquer outro site
        if n_links == 0:
            p_links = 1/n_sites
            for target in range(n_sites):
                rows.append(site)
                cols.append(target)
                data.append(p_links)
        else:
            #Probablidade de clicar em algum link referenciado
            p_links = 1/n_links
            for target in links:
                rows.append(site)
                cols.append(target)
                data.append(p_links)

        #Modifica a matriz P de tal forma a remover os zeros, mas manter sua informação
        #Isso é importante para a economia de RAM
        P = csr_matrix((data, (rows, cols)), shape=(n_sites, n_sites))

    return P

### Encontrando o estado estacionário via método de potências.

A função implementa o Método das Potências para determinar o vetor de estado estacionário $\pi$. Este vetor representa a distribuição de probabilidade de longo prazo do usuário na cadeia de Markov, que, no contexto do PageRank, corresponde ao autovetor principal da Matriz Google associado ao autovalor λ=1.

O processo simula a navegação do usuário através da multiplicação sucessiva, definida pela relação de estado:

$\pi_{n+1} = \pi_{n}P$


Onde P é a matriz de probabilidades (matriz de transição estocástica), calculada no passo anterior, e $\pi$ representa a probabilidade de estar em um site no momento $n$. (vetor de estados).

O critério de parada ocorre quando o sistema converge, ou seja, $\pi_{n+1} \approx \pi_{n}$


### Matriz Google
Ao utilizar o método das potências, é possível que erros aconteçam. O algoritmo pode ficar preso em autoloops ou em um subconjunto da cadeia, onde os sites referenciam uns aos outros, de modo a criar ciclos. Dessa forma, é introduzido um fator de amortecimento na matriz de transição estocástica para simular o comportamento aleatório do usuário de estar navegando de uma página para outra conectadas na cadeia e, repentinamente, clicar em outro site qualquer aleatoriamente. A expressão que representa esse comportamento é mostrada abaixo:


$G = \alpha*P + (1-\alpha)*\frac{1}{N}$

Na expressão, $\alpha$ representa a probabilidade do usuário continuar normalmente o percurso pela cadeia.

Para o problema, escolhemos arbitrariamente $\alpha = 0.85$






In [ ]:
#Encontra o estado estacionário da Cadeia de Markov
def find_stationary_state_by_iteraction(graph, prob_matrix):
    n_sites = len(graph)

    #Estado inicial: chances iguais de começar em qualquer site
    current_state = np.ones(n_sites)/n_sites
    P = prob_matrix

    #Definindo o valor do threshold
    threshold = 0.001
    alpha = 0.85
    teleport_constant = (1 - alpha) / n_sites


    #Número de iterações máxima: 100
    for i in range(1000):
        #Aplicando a matriz Google
        link_follow = alpha * (current_state @ P)
        next_state = link_follow + teleport_constant
        residual_value = np.linalg.norm(next_state - current_state)

        if residual_value < threshold:
            return next_state

        current_state = next_state

    print("Não foi possível encontrar um estado estacionário para esta cadeia de Markov")

### Encontrando o estado estacionário via autovetores

Similar a função descrita anteriormente, aqui calculamos o vetor de estados no estado estacionário, mas usando autovetores.

Dada a expressão:

(1) $\pi_{n+1} = \pi_{n}P$

Podemos perceber que se trata basicamente de uma transformação linear, que segue a seguinte propriedade.

(2) $T(v) = \lambda*v$

Ou seja ao rearranjarmos a expressão (1), o problema se resume a encontrar os autovetores associados ao autovalor $\lambda = 1$

$\pi_{n+1} = P^{T}\pi_{n}$





In [ ]:
#Encontra o estado estacionário da cadeia de Markov através de autovalores e autovetores
def find_stationary_state_by_eig(graph, prob_matrix):
    n_sites = len(graph)


    current_state = np.ones(n_sites)/n_sites
    P = prob_matrix

    #Encontrando autovalores e autovetores
    eigenvalues, eigenvectors = np.linalg.eig(P.T)

    #Encontrando o index do autovalor 1
    threshold = 0.0001
    eigenvec_idx = 0

    for idx, value in enumerate(eigenvalues):
        residual_value = np.abs(1 - value)
        if residual_value < threshold:
            eigenvec_idx = idx

    #Encontrar autovetor associado ao autovalor 1
    selected_eigenvec = eigenvectors[:, eigenvec_idx].real

    #Normaliza os valores, pois a soma dos resultados deve ser 1
    eigenvec_norm = selected_eigenvec/sum(selected_eigenvec)

    return eigenvec_norm

Dado que o método de potências, na maior parte das vezes, converge mais rapidamente que os método dos autovetores, além de exigir menor poder computacional, optamos por não utilizar a função find_stationary_state_by_eig() na busca final.

### Fazendo a busca e ordenando as páginas encontradas com o PageRank

Aqui recebemos uma *query* que pode ser composto por várias palavras. Usamos o índice invertido para encontrar os IDs dos artigos que contém TODAS as palavras da *query*. Depois, ordenamos a partir do ranking gerado pelo PageRank.

In [ ]:
# Faz uma busca no índice invertido e ordena o resultado com score do PageRank
def search(query, inverted_index, pagerank_vector, id_to_url):
    terms = query.lower().split()
    if not terms: return []

    # Encontrar os IDs (artigos) que tem a primeira palavra
    possible_ids = set(inverted_index.get(terms[0], []))

    # Mantém só IDs que têm TODAS as palavras
    for term in terms[1:]:
        term_ids = set(inverted_index.get(term, []))
        possible_ids = possible_ids.intersection(term_ids)
        if not possible_ids: break

    if not possible_ids:
        return []

    # Faz o ranqueamento com o PageRank
    ranked_results = []
    for doc_id in possible_ids:
        score = pagerank_vector[doc_id]
        ranked_results.append((doc_id, score))

    # Ordena do maior score para o menor
    ranked_results.sort(key=lambda x: x[1], reverse=True)

    # Formata o resultado final
    results = []
    for doc_id, score in ranked_results:
        results.append({'url': id_to_url[doc_id], 'score': score})

    return results

### Execução completa da Search Engine

Essa é função principal que realiza a busca por um termo e retorna as páginas mais adequadas.

In [ ]:
# --- BLOCO DE INTEGRAÇÃO ---

def main():
  # Carrega os dados do JSON
  print("Carregando grafo do arquivo...")
  crawler_loader = WebGraphCrawler()
  urls, adj_list, inverted_index = crawler_loader.load_graph('grafo.json')

  if urls:
      # Constroi a matriz de probabilidades P
      P = prob_matrix(adj_list)

      # Calcula o vetor de Estado Estacionário usando método das potências
      pagerank_vector = find_stationary_state_by_iteraction(adj_list, P)

      # Faz a busca pelo termo abaixo
      termo_busca = input("Digite o que você deseja pesquisar: ")

      print(f"\n--- Resultados da busca por: '{termo_busca}' ---")
      resultados = search(termo_busca, inverted_index, pagerank_vector, urls)

      if resultados:
          for i, res in enumerate(resultados[:15]): # Mostra o top 15
              print(f"{i+1:02d}. [Score: {res['score']:.6f}] {res['url']}")
      else:
          print("Nenhum resultado encontrado para este termo.")

  else:
      print("Erro: Grafo não pôde ser carregado. Verifique se o arquivo existe.")

if __name__ == "__main__":
  main()

Carregando grafo do arquivo...
Grafo carregado de grafo.json. Total URLs: 2282
Digite o que você deseja pesquisar: álgebra linear

--- Resultados da busca por: 'álgebra linear' ---
01. [Score: 0.000427] https://pt.wikipedia.org/wiki/Mec%C3%A2nica_qu%C3%A2ntica
02. [Score: 0.000423] https://pt.wikipedia.org/wiki/Geometria
03. [Score: 0.000417] https://pt.wikipedia.org/wiki/Matem%C3%A1tico
